# DomesticDeclarations Dataset Loader

This notebook loads the DomesticDeclarations XES dataset, converts it to the CSV format expected by EventLogLoader, and creates the encoded train/val/test datasets.

In [ ]:
import importlib
import sys
import torch
import numpy as np
import pandas as pd
import pm4py
from pathlib import Path

_current = Path().resolve()
while _current != _current.parent:
    if (_current / 'src').is_dir() and (_current / 'data').is_dir():
        break
    _current = _current.parent
_project_root = _current

for p in [_project_root, _project_root / 'src']:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

## 1. Load XES File and Convert to CSV Format

In [ ]:
# Load XES file using pm4py
xes_path = str(_project_root / 'data' / 'DomesticDeclarations.xes')
log = pm4py.read_xes(xes_path)
df_xes = pm4py.convert_to_dataframe(log)

print(f"Loaded {len(df_xes)} events from {len(df_xes['case:concept:name'].unique())} cases")
print(f"Columns: {df_xes.columns.tolist()}")
df_xes.head()

In [ ]:
# pm4py already provides columns: Case ID, Activity, Complete Timestamp, Resource, Role, Amount
# Select only the columns we need (dropping raw XES columns like concept:name, time:timestamp, etc.)
columns_to_keep = ['Case ID', 'Activity', 'Complete Timestamp', 'Resource', 'Role', 'Amount']
df_csv = df_xes[columns_to_keep].copy()

# Simplify Activity: extract base action from composite activity name.
# The Role column already captures the role part, so we reduce Activity
# from 17 composite values to 8 base actions.
def simplify_activity(act: str) -> str:
    """Extract base action from composite activity name.
    'Declaration SUBMITTED by EMPLOYEE' → 'SUBMITTED'
    'Request Payment' → 'Request Payment' (unchanged)
    """
    if act.startswith('Declaration ') and ' by ' in act:
        return act.split('Declaration ')[1].split(' by ')[0]
    return act

df_csv['Activity'] = df_csv['Activity'].map(simplify_activity)

# Convert timestamp to string format expected by EventLogLoader
df_csv['Complete Timestamp'] = df_csv['Complete Timestamp'].dt.strftime('%Y/%m/%d %H:%M:%S.%f')

print(f"Converted DataFrame shape: {df_csv.shape}")
print(f"\nColumn types:")
print(df_csv.dtypes)
print(f"\nUnique activities: {df_csv['Activity'].nunique()}")
print(df_csv['Activity'].unique())
df_csv.head(10)

In [ ]:
# Save to CSV for EventLogLoader (lands in project-root data/)
csv_path = str(_project_root / 'data' / 'domestic_declarations.csv')
df_csv.to_csv(csv_path, index=False)
print(f"Saved CSV to {csv_path}")

## 2. Load and Encode Data Using EventLogLoader

In [ ]:
import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogLoader, EventLogDataset

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

np.random.seed(17)

event_log_location = str(_project_root / 'data' / 'domestic_declarations.csv')

result_name = 'domestic_declarations_all'

event_log_properties = {
    'case_name' : 'Case ID',
    'concept_name' : 'Activity',
    'timestamp_name' : 'Complete Timestamp',
    'date_format' : '%Y/%m/%d %H:%M:%S.%f',
    'time_since_case_start_column' : 'case_elapsed_time',
    'time_since_last_event_column' : 'event_elapsed_time',
    'day_in_week_column' : 'day_in_week',
    'seconds_in_day_column' : 'seconds_in_day',
    'min_suffix_size' : 5,
    'train_validation_size' : 0.15,
    'test_validation_size' : 0.2,
    'window_size' : 'auto',
    'categorical_columns' : ['Activity', 'Resource', 'Role'],
    'continuous_columns' : ['case_elapsed_time', 'event_elapsed_time', 'day_in_week', 'seconds_in_day', 'Amount'],
    'continuous_positive_columns' : [],
}


# 1) loads event log
# 2) adds EOS to cases
# 3) normalizes numerical features
# 4) imputes features (replaces n.a. categories with a new class for categorical features and average values for numerical features)
event_log_loader = EventLogLoader(event_log_location, event_log_properties)

In [ ]:
print(f"Calculated window size: {event_log_loader.encoder_decoder.window_size}")

Calculated window size: 20


In [ ]:
# Explore the data
print(f"Number of unique cases: {len(event_log_loader.event_log.df['Case ID'].unique())}")
print(f"\nCase length distribution:")
case_lengths = event_log_loader.event_log.df.groupby('Case ID').size()
print(case_lengths.describe())

Number of unique cases: 10500

Case length distribution:
count    10500.000000
mean        10.374952
std          1.486415
min          6.000000
25%         10.000000
50%         10.000000
75%         11.000000
max         29.000000
dtype: float64


## 3. Save Encoded Datasets

In [ ]:
!ls .

README.md     philipp       temporary     test_philipp  test_weytjens


In [ ]:
train_dataset = event_log_loader.get_dataset('train')
torch.save(train_dataset, ''+result_name+'_'+str(event_log_loader.encoder_decoder.min_suffix_size)+'_train.pkl')
print(f"Train dataset saved with {len(train_dataset)} samples")
print(f"Categories: {train_dataset.all_categories}")

categorical tensors:   0%|          | 0/3 [00:00<?, ?it/s]

Activity:   0%|          | 0/6825 [00:00<?, ?it/s]

Resource:   0%|          | 0/6825 [00:00<?, ?it/s]

Role:   0%|          | 0/6825 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/5 [00:00<?, ?it/s]

...

event_elapsed_time:   0%|          | 0/6825 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/6825 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/6825 [00:00<?, ?it/s]

Amount:   0%|          | 0/6825 [00:00<?, ?it/s]

Train dataset saved with 50361 samples
Categories: ([('Activity', 18, {'Declaration APPROVED by ADMINISTRATION': 1, 'Declaration APPROVED by BUDGET OWNER': 2, 'Declaration APPROVED by PRE_APPROVER': 3, 'Declaration FINAL_APPROVED by SUPERVISOR': 4, 'Declaration FOR_APPROVAL by ADMINISTRATION': 5, 'Declaration FOR_APPROVAL by SUPERVISOR': 6, 'Declaration REJECTED by ADMINISTRATION': 7, 'Declaration REJECTED by BUDGET OWNER': 8, 'Declaration REJECTED by EMPLOYEE': 9, 'Declaration REJECTED by MISSING': 10, 'Declaration REJECTED by PRE_APPROVER': 11, 'Declaration REJECTED by SUPERVISOR': 12, 'Declaration SAVED by EMPLOYEE': 13, 'Declaration SUBMITTED by EMPLOYEE': 14, 'EOS': 15, 'Payment Handled': 16, 'Request Payment': 17}), ('Resource', 4, {'EOS': 1, 'STAFF MEMBER': 2, 'SYSTEM': 3}), ('Role', 9, {'ADMINISTRATION': 1, 'BUDGET OWNER': 2, 'EMPLOYEE': 3, 'EOS': 4, 'MISSING': 5, 'PRE_APPROVER': 6, 'SUPERVISOR': 7, 'UNDEFINED': 8})], [('case_elapsed_time', 1, {}), ('event_elapsed_time', 1, {})

In [ ]:
test_dataset = event_log_loader.get_dataset('test')
torch.save(test_dataset, ''+result_name+'_'+str(event_log_loader.encoder_decoder.min_suffix_size)+'_test.pkl')
print(f"Test dataset saved with {len(test_dataset)} samples")

categorical tensors:   0%|          | 0/3 [00:00<?, ?it/s]

Activity:   0%|          | 0/2100 [00:00<?, ?it/s]

Resource:   0%|          | 0/2100 [00:00<?, ?it/s]

Role:   0%|          | 0/2100 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/5 [00:00<?, ?it/s]

...

event_elapsed_time:   0%|          | 0/2100 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/2100 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/2100 [00:00<?, ?it/s]

Amount:   0%|          | 0/2100 [00:00<?, ?it/s]

Test dataset saved with 15471 samples


In [ ]:
val_dataset = event_log_loader.get_dataset('val')
torch.save(val_dataset, ''+result_name+'_'+str(event_log_loader.encoder_decoder.min_suffix_size)+'_val.pkl')
print(f"Validation dataset saved with {len(val_dataset)} samples")

categorical tensors:   0%|          | 0/3 [00:00<?, ?it/s]

Activity:   0%|          | 0/1575 [00:00<?, ?it/s]

Resource:   0%|          | 0/1575 [00:00<?, ?it/s]

Role:   0%|          | 0/1575 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/5 [00:00<?, ?it/s]

...

event_elapsed_time:   0%|          | 0/1575 [00:00<?, ?it/s]

day_in_week:   0%|          | 0/1575 [00:00<?, ?it/s]

seconds_in_day:   0%|          | 0/1575 [00:00<?, ?it/s]

Amount:   0%|          | 0/1575 [00:00<?, ?it/s]

Validation dataset saved with 11605 samples


In [ ]:
# Verify saved datasets
print("\n=== Dataset Summary ===")
print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\nWindow size: {event_log_loader.encoder_decoder.window_size}")
print(f"Min suffix size: {event_log_loader.encoder_decoder.min_suffix_size}")


=== Dataset Summary ===
Train samples: 50361
Validation samples: 11605
Test samples: 15471

Window size: 20
Min suffix size: 5
